In [2]:
import pyspark
import sparkmonitor
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [3]:
listener_jar = Path(sparkmonitor.__path__[0]) / "listener_spark4_2.13.jar"
spark = (SparkSession.builder
    .config(
        "spark.extraListeners",
        "sparkmonitor.listener.JupyterSparkMonitorListener",
    )
    .config("spark.driver.extraClassPath", str(listener_jar))
    .appName("PySpark-Get-Started")
    .getOrCreate()
)

In [4]:
df_csv = (
            spark.read.csv("data/orders.csv",
                           header=True,
                          )
)

In [5]:
df_csv.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- price: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- shipping_address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- region: string (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- delivery_days: string (nullable = true)
 |-- returned: string (nullable = true)
 |-- gender: string (nullable = true)



In [6]:
df_csv.explain()

== Physical Plan ==
FileScan csv [order_id#17,customer_id#18,order_date#19,product_id#20,quantity#21,price#22,order_status#23,shipping_address#24,city#25,country#26,payment_method#27,discount#28,category#29,sales_rep#30,region#31,ship_date#32,delivery_days#33,returned#34,gender#35] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/ggrft/jupyter-lab/apache-spark/udemy/data/orders.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,customer_id:string,order_date:string,product_id:string,quantity:string,pri...




In [7]:
#Explain reparitions
df_repartition = df_csv.repartition(4)
df_repartition.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=28]
   +- FileScan csv [order_id#17,customer_id#18,order_date#19,product_id#20,quantity#21,price#22,order_status#23,shipping_address#24,city#25,country#26,payment_method#27,discount#28,category#29,sales_rep#30,region#31,ship_date#32,delivery_days#33,returned#34,gender#35] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/ggrft/jupyter-lab/apache-spark/udemy/data/orders.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,customer_id:string,order_date:string,product_id:string,quantity:string,pri...




In [10]:
# For coalesce
df_coalesce = df_csv.coalesce(2)
df_coalesce.explain()

== Physical Plan ==
Coalesce 2
+- FileScan csv [order_id#17,customer_id#18,order_date#19,product_id#20,quantity#21,price#22,order_status#23,shipping_address#24,city#25,country#26,payment_method#27,discount#28,category#29,sales_rep#30,region#31,ship_date#32,delivery_days#33,returned#34,gender#35] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/ggrft/jupyter-lab/apache-spark/udemy/data/orders.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,customer_id:string,order_date:string,product_id:string,quantity:string,pri...




In [12]:
# withColumn
df_wc = df_csv.withColumn("more_discount", col("discount")*1.02)
df_wc.explain()
## "Project" is for select and withColumn

== Physical Plan ==
*(1) Project [order_id#17, customer_id#18, order_date#19, product_id#20, quantity#21, price#22, order_status#23, shipping_address#24, city#25, country#26, payment_method#27, discount#28, category#29, sales_rep#30, region#31, ship_date#32, delivery_days#33, returned#34, gender#35, (cast(discount#28 as double) * 1.02) AS more_discount#60]
+- FileScan csv [order_id#17,customer_id#18,order_date#19,product_id#20,quantity#21,price#22,order_status#23,shipping_address#24,city#25,country#26,payment_method#27,discount#28,category#29,sales_rep#30,region#31,ship_date#32,delivery_days#33,returned#34,gender#35] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/ggrft/jupyter-lab/apache-spark/udemy/data/orders.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,customer_id:string,order_date:string,product_id:string,quantity:string,pri...




In [11]:
# groupby
df_groupby = df_csv.groupBy("product_id").count()
df_groupby.explain()

## Two  HashAggregate because on is a partial local agg count ( functions=[partial_count(1)] )
## , before final merge count (functions=[count(1)])

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[product_id#20], functions=[count(1)])
   +- Exchange hashpartitioning(product_id#20, 200), ENSURE_REQUIREMENTS, [plan_id=67]
      +- HashAggregate(keys=[product_id#20], functions=[partial_count(1)])
         +- FileScan csv [product_id#20] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/ggrft/jupyter-lab/apache-spark/udemy/data/orders.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<product_id:string>




In [13]:
spark.stop()